# 13. Calling the Jira REST API with requests

The `jira` library covers the common tasks. For anything it doesn't wrap, or to understand what happens underneath, call the REST API directly with **`requests`**.

| Part | In this notebook |
|---|---|
| **URL** | `https://your-domain.atlassian.net/rest/api/2/...` |
| **Method** | `requests.get()` to read, `requests.post()` to create |
| **Headers** | `Accept` and `Content-Type: application/json` |
| **Auth** | `auth=(email, api_token)` |
| **Payload** | The JSON body for POST requests |

> **Before you run this notebook:** create an API token (topic 08) and set `JIRA_SERVER`, `JIRA_EMAIL` and `JIRA_API_TOKEN` as environment variables, or in Colab's **Secrets** panel. The outputs shown were recorded on a practice Jira Cloud site with a project called **myproject** (key `MYP`). Your issue keys and data will differ.

In [1]:
import json
import os
import requests

server = os.environ["JIRA_SERVER"]
user = os.environ["JIRA_EMAIL"]
my_token = os.environ["JIRA_API_TOKEN"]

headers = {"Accept": "application/json", "Content-Type": "application/json"}

## Without Authentication: 401

Ask for a user's profile without credentials:

In [2]:
account_id = "<your-account-id>"        # from jira.search_users() in topic 12
url = f"{server}/rest/api/2/user?accountId={account_id}"

response = requests.get(url, headers=headers)
my_status_code = response.status_code
print(my_status_code)
if response.status_code == 200:
    print("Success", my_status_code)
else:
    print("Error", my_status_code)

401
Error 401


**401 Unauthorized** means the request reached Jira but wasn't allowed. Common causes:

- No credentials, or a wrong email or token
- An expired or revoked API token
- The account has no permission for that resource

## With Authentication: 200

In [3]:
response = requests.get(url, headers=headers, auth=(user, my_token))
my_status_code = response.status_code
print(my_status_code)
if response.status_code == 200:
    print("Success", my_status_code)
else:
    print("Error", my_status_code)

200
Success 200


## Reading the JSON: an Email from an Account ID

In [4]:
my_data = response.json()
my_data["emailAddress"]

'<your-email>'

> Jira only returns `emailAddress` when the user's profile visibility allows it. Many accounts hide their email, so code that needs it should handle a missing key: `my_data.get("emailAddress")`.

## Reading an Issue

In [5]:
response = requests.get(f"{server}/rest/api/2/issue/MYP-8", headers=headers, auth=(user, my_token))
issue = response.json()
print(issue["key"], "|", issue["fields"]["summary"], "|", issue["fields"]["status"]["name"])

## Creating an Issue with POST

A POST request sends a **payload**: the JSON data the server should store.

- **Headers** are like the label on an envelope: they say how to read the contents.
- The **payload** is the letter inside.

In [6]:
url = f"{server}/rest/api/3/issue"

issue_dict = {
    "fields": {
        "project": {"key": "MYP"},
        "issuetype": {"name": "Task"},
        "summary": "Create issue from Python",
        "priority": {"name": "High"},
    }
}

payload = json.dumps(issue_dict)
response = requests.post(url, headers=headers, auth=(user, my_token), data=payload)
print(response.json())

{'id': '10098', 'key': 'MYP-76', 'self': 'https://your-domain.atlassian.net/rest/api/3/issue/10098'}


`json.dumps()` turns the Python dictionary into a JSON string. You can also pass `json=issue_dict` and let `requests` do it.

### A Description in Atlassian Document Format

API **v3** stores rich text as **ADF**, a nested JSON document. A plain string in `description` is rejected by v3, so wrap it:

In [7]:
issue_dict = {
    "fields": {
        "project": {"key": "MYP"},
        "issuetype": {"name": "Task"},
        "summary": "Create issue from Python",
        "description": {
            "type": "doc",
            "version": 1,
            "content": [
                {"type": "paragraph", "content": [{"type": "text", "text": "Create issue from Python"}]}
            ],
        },
        "priority": {"name": "High"},
    }
}

response = requests.post(url, headers=headers, auth=(user, my_token), json=issue_dict)

if response.status_code == 201:
    print("Issue created successfully!")
    print(response.json())
else:
    print(f"Failed to create issue. Status code: {response.status_code}")
    print(response.json())

Issue created successfully!
{'id': '10097', 'key': 'MYP-75', 'self': 'https://your-domain.atlassian.net/rest/api/3/issue/10097'}


> With API **v2** (`/rest/api/2/issue`), `description` can stay a plain string. That's what the `jira` library uses.

## Status Codes You'll See

| Code | Meaning | Typical cause |
|---|---|---|
| 200 | OK | A successful GET |
| 201 | Created | A successful POST that created an issue |
| 204 | No Content | A successful update or delete |
| 400 | Bad Request | A wrong field name or value, such as an unknown priority |
| 401 | Unauthorized | Missing or wrong credentials |
| 403 | Forbidden | Signed in, but no permission |
| 404 | Not Found | Wrong URL, or an issue you can't see |
| 429 | Too Many Requests | Rate limited. Wait and retry |

When a request fails, print `response.json()`: Jira's error messages name the exact field that's wrong.